# Price per Square Meter and Location Effects

**Question:** Are property prices in Mexico more influenced by property size or by location?

This is the last notebook. I use price per square meter to compare states and property types,
and I measure how much of the price variation comes from location.

## 1. Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. New column: price per square meter

In [ ]:
# Create new column 'price_per_m2'
df = (
    pd.read_csv("./data/mexico-real-estate-combined-clean.csv")
    .assign(price_per_m2 = lambda x: x['price_usd'] / x['area_m2'])
)

# Display basic statistics of the new column
print("Price per m2 Statistics:")
print(df["price_per_m2"].describe())

df.head()

## 3. Distribution of price per square meter

In [ ]:
# Create figure and axes
fig, ax = plt.subplots()

# Create histogram with 30 bins
ax.hist(df["price_per_m2"], bins=30, edgecolor="white")

# Set labels and title
ax.set_title("Distribution of Price per Square Meter")
ax.set_xlabel("Price per m² [USD]")
ax.set_ylabel("Frequency")

# Add a vertical line for the median
median_price_per_m2 = df["price_per_m2"].median()
ax.axvline(median_price_per_m2, color='red', linestyle='--',
           label=f'Median: ${median_price_per_m2:.2f}/m²')
ax.legend()

plt.tight_layout()
plt.show()

## 4. The location effect: comparing states

### 4.1 Statistics for each state

In [ ]:
# Calculate statistics by state
state_price_stats = (
    df
    .groupby("state")
    ["price_per_m2"]                    # <--- "price_per_m2"
    .agg(['mean', 'median', 'std'])
    .sort_values(by='median', ascending=False)

    # Round to 2 decimal places for readability
    .round(2)
)


print("Price per m2 Statistics by State:")
state_price_stats

### 4.2 Most vs least expensive states

In [ ]:
# 1. Identify the states we want to plot (Top 3 and Bottom 3)
top_states = state_price_stats.head(3).index.tolist()
bottom_states = state_price_stats.tail(3).index.tolist()
target_states = top_states + bottom_states

# 2. Filter data by target_states
df_states = df[df["state"].isin(target_states)]

# 3. Prepare data for plotting
# Create a list of arrays, one for each state in our target list
data_to_plot = [df_states[df_states["state"] == state]["price_per_m2"]
                for state in target_states]

# 4. Plot
fig, ax = plt.subplots()

ax.boxplot(data_to_plot, tick_labels=target_states, vert=False)
ax.set_title("Price per m² Distribution: Most vs. Least Expensive States")
ax.set_xlabel("Price per m² [USD]")
ax.set_ylabel("State")

plt.tight_layout()
plt.show()

### 4.3 How much of the variation comes from location?

I split the variation in price into two parts: differences **between** states,
and differences **within** each state.

In [ ]:
# 1. Correlation between size and price
size_price_correlation = df[["area_m2", "price_usd"]].corr().iloc[0, 1]
print(f"Correlation between area and price: {size_price_correlation:.3f}")

# 2. Calculate variance components
# Total variance in price_usd
total_variance = df["price_usd"].var()

# Between-state variance (variance of state_means)
state_means = df.groupby("state")["price_usd"].mean()
between_variance = state_means.var()

# Within-state variance (average variance within each state)
within_variance = df.groupby("state")["price_usd"].var().mean()

# 3. Calculate percentages
between_pct = (between_variance / total_variance) * 100
within_pct = (within_variance / total_variance) * 100

print(f"\nVariance Decomposition:")
print(f"  Between-state variance: {between_pct:.1f}%")
print(f"  Within-state variance: {within_pct:.1f}%")
print(f"  Ratio (Between/Within): {between_variance/within_variance:.2f}")

## 5. The property type effect

In [ ]:
# 1. Calculate summary statistics by property_type
property_type_stats = (
    df
    .groupby("property_type")
    ["price_per_m2"]      #   <--- "price_per_m2"
    .agg(['mean', 'median', 'count'])
    .round(2)
)

print("Price per m² by Property Type:")
print(property_type_stats)

# 2. Get unique property types
prop_types = df["property_type"].unique()

# 3. Prepare data list
data_by_type = [df[df["property_type"] == p_type]["price_per_m2"]
                for p_type in prop_types]

# 4. Plot
fig, ax = plt.subplots()

ax.boxplot(data_by_type, tick_labels=prop_types, vert=False)
ax.set_title("Price per m² by Property Type")
ax.set_xlabel("Price per m² [USD]")
ax.set_ylabel("Property Type")

plt.tight_layout()
plt.show()

## 6. State and property type together
I only keep groups with at least 5 listings, so small groups do not give misleading results.

In [ ]:
# 1. Group by both state and property_type
combined_stats = (
    df
    .groupby(["state", "property_type"])   # <-- "state" and "property_type"
    ["price_per_m2"]
    .agg(['median', 'count'])
    .round(2)
)

# 2. Filter for groups with at least 5 properties
combined_stats_filtered = combined_stats[combined_stats['count'] >= 5]

# 3. Sort by median price per m²
combined_stats_filtered = combined_stats_filtered.sort_values(
    by='median', ascending=False
)

print("Top 15 State-Property Type Combinations:")
display(combined_stats_filtered.head(15))

print("\nBottom 10 State-Property Type Combinations:")
combined_stats_filtered.tail(10)